### Reference
https://www.kaggle.com/maxwellgreene/r-titanic-lm-glm-randomforest
https://www.kaggle.com/suvedhand/getting-started-with-titanic-r
https://www.kaggle.com/mrisdal/exploring-survival-on-the-titanic

## Initialize and Import

In [ ]:
knitr::opts_chunk$set(fig.width = 7, fig.height = 4.5, 
                      warning = TRUE, echo = TRUE)

In [ ]:
# Load packages
library(tidyverse)  # metapackage of all tidyverse packages
library(GGally)  # visualization
library(mice)  # PMM(Predictive Mean Matching)
library('randomForest') # classification algorithm


## Load and check data

In [ ]:

# List the competition files
cat("Data files available: ",list.files(path = "../input",recursive=TRUE),sep = "\n \t")


In [ ]:

# Read Data & bind
train <- read.csv('../input/titanic/train.csv', stringsAsFactors = F)
test  <- read.csv('../input/titanic/test.csv', stringsAsFactors = F)

full  <- bind_rows(train, test) # bind training & test data

# check data
print("full data")
str(full)


There is data for 1309 people in total.Survived NA = number of test data.  

Variable Name | Description
--------------|-------------
Survived      | Survived (1) or died (0)
Pclass        | Passenger's class
Name          | Passenger's name
Sex           | Passenger's sex
Age           | Passenger's age
SibSp         | Number of siblings/spouses aboard
Parch         | Number of parents/children aboard
Ticket        | Ticket number
Fare          | Fare
Cabin         | Cabin
Embarked      | Port of embarkation


Getting rid of some variables we won't need and converting appropriate characters to factors.

In [ ]:

# Remove data
full$PassengerId <- NULL
full$Name        <- NULL
full$Ticket      <- NULL
full$Cabin       <- NULL

#Change to string-> factor type
full$Survived      <- as.factor(full$Survived)
full$Pclass      <- as.factor(full$Pclass)
full$Sex         <- as.factor(full$Sex)
full$Embarked    <- as.factor(full$Embarked)

print("full data")
str(full)
summary(full)


Embarked ""'s 2, Age NA's 263, Fare NA's 1 missing  

# Data visualization & Quick Cleaning

In [ ]:

print("Number of male/female and survivals")
table(full$Survived,full$Sex)

print("Number of Pclass and survivals")
table(full$Survived,full$Pclass)

print("Number of Parch and survivals")
table(full$Survived,full$Parch)

ggplot(full,aes(x=Sex, fill = full$Survived)) +
  geom_bar( position = "fill")

ggplot(full,aes(x=Pclass, fill = Survived)) +
  geom_bar( position = "fill")

ggplot(full,aes(x=Parch, fill = Survived)) +
  geom_bar( position = "fill")

ggplot(full) +
  geom_histogram( aes(x = Age,
                  fill = Survived),
                  alpha = 0.8,
                  binwidth = 10)

ggplot(full) +
  geom_histogram( aes(x = Fare,
                  fill = Survived),
                  alpha = 0.8)


The mortality rate of children is low, and more men die than women.  
If the number of families is large, there are many deaths.(Parch >= 4)  
Higher fares (Fara) tend to have higher survival rates.  


## Processing to fill NA (Fare)  

In [ ]:

# Extract rows of NA data, display data.
print("Fare == NA line")
match(NA, full$Fare)
print(full[1044,])

#Visualize the relationship between Fare and other variables.

ggplot(full) +
  geom_histogram( aes(x = Fare,
                  fill = Embarked),
                  alpha = 0.5)

ggplot(full) +
  geom_boxplot( aes(y = Fare,
                  fill = Pclass),
                  alpha = 0.5)



Passenger = row[1044] is Pclass=3 and Embarked = s.  
Fill NA with median.  

In [ ]:

# Replace missing fare value with median fare for class/embarkment
full$Fare[1044] <- median(full[full$Pclass == '3' &
                          full$Embarked == 'S',]$Fare, na.rm = TRUE)
print(full[1044,])



## Processing to fill NA(Age)

In [ ]:

# Show number of missing Age values
sum(is.na(full$Age))


In [ ]:

#PMM Method
mice_mod <-
  mice(full[,-1], m = 10, maxit = 50, method = "pmm", seed = 1912)

mice_output <- complete(mice_mod)


In [ ]:
# Plot age distributions
par(mfrow=c(1,2))
hist(full$Age, freq=F, main='Age: Original Data', 
  col='darkgreen', ylim=c(0,0.04))
hist(mice_output$Age, freq=F, main='Age: MICE Output', 
  col='lightgreen', ylim=c(0,0.04))

Let’s compare the results we get with the original distribution of passenger ages to ensure that nothing has gone completely awry.  


In [ ]:
# Replace Age variable from the mice model.
full$Age <- mice_output$Age

# Show new number of missing Age values
sum(is.na(full$Age))

## Processing to fill NA(Embarked)

In [ ]:

print("Embarked == NA line")
filter(full, full$Embarked == "")


# Use ggplot2 to visualize embarkment, passenger class, & median fare
ggplot(full, aes(x = Embarked, y = Fare, fill = factor(Pclass))) +
  geom_boxplot() +
  geom_hline(aes(yintercept=80), 
    colour='red', linetype='dashed', lwd=2) 

# Passengers 62 and 830 are missing Embarkment
full[c(62, 830), 'Embarked']

sum(is.na(full$Embarked))


## Split into training & test sets

Split the data back into the original test and training sets.  

In [ ]:
# Split the data back into a train set and a test set
train <- full[1:891,]
test <- full[892:1309,]

# Building the model 


## General Linear Model(Logistic)

Predict with all variables and exclude non-significant variables.  

In [ ]:

glm1 <- glm(Survived ~ ., 
            family = "binomial", data = train)

#glm1_train <- as.integer(predict(glm1,train) > -0.25)
#glm1_test  <- as.integer(predict(glm1,test) > -0.25)

#testontrain(glm1,type="glm")
summary(glm1)
library(coefplot)
coefplot::coefplot(glm1)


In [ ]:

glm2 <- glm(Survived ~ Sex + Age + Fare + Parch +SibSp +Pclass, 
            family = "binomial", data = train)

#glm1_train <- as.integer(predict(glm1,train) > -0.25)
#glm1_test  <- as.integer(predict(glm1,test) > -0.25)

#testontrain(glm1,type="glm")
summary(glm2)
coefplot::coefplot(glm2)


In [ ]:

glm3 <- glm(Survived ~ Sex + SibSp + Pclass, 
            family = "binomial", data = train)

#glm1_train <- as.integer(predict(glm1,train) > -0.25)
#glm1_test  <- as.integer(predict(glm1,test) > -0.25)

#testontrain(glm1,type="glm")
summary(glm3)
coefplot::coefplot(glm3)



## ROC curve and AUC(Logistic)

In [ ]:

# ROC curve and AUC
library(pROC)

train_read <- read.csv('../input/titanic/train.csv', stringsAsFactors = F)
train_glm <- predict(glm3,newdata=train, type="response")

glm_mat <- table(ifelse(train_glm>=0.5,1,0), train$Survived)
glm_mat

roc_glm <- roc(response = train_read$Survived, predictor = train_glm)
plot(roc_glm, legacy.axes = TRUE)

auc(roc_glm)


## Random Forest

In [ ]:
# Set a random seed
set.seed(1912)

# Build the model (note: not all possible variables are used)
rf_model <- randomForest(factor(Survived) ~ Pclass + Sex + Age + SibSp + Parch + 
                                            Fare + Embarked ,
                                            data = train)

# Show model error
plot(rf_model, ylim=c(0,0.36))
legend('topright', colnames(rf_model$err.rate), col=1:3, fill=1:3)


# Variable importance

Look at relative variable importance by plotting the mean decrease in Gini calculated across all trees.

In [ ]:
# Get importance
importance    <- importance(rf_model)
varImportance <- data.frame(Variables = row.names(importance), 
                            Importance = round(importance[ ,'MeanDecreaseGini'],2))

# Create a rank variable based on importance
rankImportance <- varImportance %>%
  mutate(Rank = paste0('#',dense_rank(desc(Importance))))

# Use ggplot2 to visualize the relative importance of variables
ggplot(rankImportance, aes(x = reorder(Variables, Importance), 
    y = Importance, fill = Importance)) +
  geom_bar(stat='identity') + 
  geom_text(aes(x = Variables, y = 0.5, label = Rank),
    hjust=0, vjust=0.55, size = 4, colour = 'red') +
  labs(x = 'Variables') +
  coord_flip() 

## ROC curve and AUC(Random Forest)

In [ ]:

# ROC curve and AUC

train_lf <- predict(rf_model, train)

lf_mat <- table(train_lf, train$Survived)
lf_mat

train_lf_p <- predict(rf_model,train,type='prob')
train_lf_prob <- as.numeric(train_lf_p[,1]) 

roc_lf <- roc(response = train_read$Survived, predictor = train_lf_prob,
              ci = TRUE)
plot(roc_lf, legacy.axes = TRUE)

auc(roc_lf)


# Prediction


In [ ]:

# Apply the predictive model to the test data.
# Predict using the test set @glm3(Logistic)
# Survival: 1 if survival probability is 0.5 or more, death: 0 otherwise.
solution_glm <- ifelse(predict(glm3,newdata=test, type="response")>=0.5,1,0)

test_read <- read.csv('../input/titanic/test.csv', stringsAsFactors = F)

output = data.frame('PassengerId' = test_read$PassengerId, 
                    'Survived' = solution_glm)
head(output)
write.csv(output, file = 'glm_mod_Solution.csv', row.names = F)


# Predict using the test set @rf
solution_rf <- predict(rf_model, test)

output = data.frame('PassengerId' = test_read$PassengerId, 
                    'Survived' = solution_rf)
head(output)
write.csv(output, file = 'rf_mod_Solution.csv', row.names = F)
